In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install wandb huggingface_hub datasets gradio -q

print("✅ Done. Go to Runtime → Restart session, then continue from CELL 2.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 18.8 MB/s eta 0:00:00
✅ Done. Go to Runtime → Restart session, then continue from CELL 2.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
BASE = "/content/drive/MyDrive/cp_coder"
os.makedirs(f"{BASE}/data",    exist_ok=True)
os.makedirs(f"{BASE}/outputs", exist_ok=True)
os.makedirs(f"{BASE}/scripts", exist_ok=True)
print(f"Project root: {BASE}")

# HuggingFace login (needed for Qwen gated model)
from huggingface_hub import notebook_login
notebook_login()

# W&B login
import wandb
wandb.login()


Mounted at /content/drive
Project root: /content/drive/MyDrive/cp_coder


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumar007neelesh (kumar007neelesh-iit-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
!python /content/1_prepare_data.py

[1/3] Loading dataset 'deepmind/code_contests' split='train' …
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'deepmind/code_contests' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 13.0kB [00:00, 23.1MB/s]
Resolving data files: 100% 39/39 [00:00<00:00, 26728.41it/s]
dataset_infos.json: 4.52kB [00:00, 5.73MB/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:00<?, ?B/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:00<?, ?B/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:00<?, ?B/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:00<?, ?B/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:00<?, ?B/s]
data/train-00000-of-00039-e991a271dbfa99(…):   0% 0.00/180M [00:01<?, ?B/s]
data/train-00000-of-000

In [4]:
DATA_PATH = f"{BASE}/data/cp_chatml.jsonl"

import os
if os.path.exists(DATA_PATH):
    # Count records to confirm it's usable
    with open(DATA_PATH) as f:
        count = sum(1 for _ in f)
    print(f"✅ Dataset already exists: {count:,} records at {DATA_PATH}")
    print("   Skipping data preparation. Delete the file to re-run.")

    # Delete the file if it exists and is empty or has 0 records
    if count == 0:
        os.remove(DATA_PATH)
        print(f"Deleted empty dataset: {DATA_PATH}")
        print("Dataset not found. Running 1_prepare_data.py ...")
        !python /content/1_prepare_data.py \
            --dataset deepmind/code_contests \
            --output  {DATA_PATH} \
            --max     20000

else:
    print("Dataset not found. Running 1_prepare_data.py ...")
    !python /content/1_prepare_data.py \
        --dataset deepmind/code_contests \
        --output  {DATA_PATH} \
        --max     20000


✅ Dataset already exists: 11,086 records at /content/drive/MyDrive/cp_coder/data/cp_chatml.jsonl
   Skipping data preparation. Delete the file to re-run.


In [ ]:

ADAPTER_PATH = f"{BASE}/outputs/lora_adapter"

!python /content/2_train.py \
    --model_name   Qwen/Qwen2.5-Coder-7B-Instruct \
    --data_path    {DATA_PATH} \
    --output_dir   {BASE}/outputs \
    --lora_r       8 \
    --lora_alpha   16 \
    --learning_rate 2e-4 \
    --num_epochs   3 \
    --batch_size   1 \
    --grad_accum   16 \
    --max_seq_length 1024 \
    --wandb_project cp-coder-t4 \
    --wandb_run_name qwen2.5-7b-t4-run1


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: kumar007neelesh (kumar007neelesh-iit-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run pg2oqilr (0.1s)
wandb: ⣽ setting up run pg2oqilr (0.1s)
wandb: ⣾ setting up run pg2oqilr (0.1s)
wandb: ⣷ setting up run pg2oqilr (0.1s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/wandb/run-20260427_045621-pg2oqilr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qwen2.5-7b-t4-run1
wandb: ⭐️ View project at https://wandb.ai/kumar007neelesh-iit-/cp-coder-t4
wandb: 🚀 View run at https://wandb.ai/kumar007neelesh-iit-/cp-coder-t4/runs/pg2oqilr
wandb: Detected [huggingface_hub.inference, openai] 

In [ ]:
import os
adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
all_present = all(os.path.exists(f"{ADAPTER_PATH}/{f}") for f in adapter_files)

if all_present:
    print(f"✅ LoRA adapter found at: {ADAPTER_PATH}")
    for f in os.listdir(ADAPTER_PATH):
        size_mb = os.path.getsize(f"{ADAPTER_PATH}/{f}") / 1e6
        print(f"   {f:40s} {size_mb:.1f} MB")
else:
    print("❌ Adapter files missing. Training may not have completed.")
    print(f"   Check {BASE}/outputs/ for partial checkpoints.")


In [ ]:
HF_USERNAME = "your-username"
HF_REPO     = "cp-coder-7b"
HUB_ID      = f"{HF_USERNAME}/{HF_REPO}"

import os
os.environ["HF_TOKEN"] = "hf_xxx"

!python /content/3_merge_and_export.py \
    --adapter_path {ADAPTER_PATH} \
    --base_model   Qwen/Qwen2.5-Coder-7B-Instruct \
    --hub_id       {HUB_ID} \
    --save_local \
    --local_dir    {BASE}/outputs \
    --export_gguf \
    --gguf_quant   q4_k_m

print(f"\n✅ Model live at: https://huggingface.co/{HUB_ID}")
print(f"✅ GGUF live at:  https://huggingface.co/{HUB_ID}-GGUF")


In [ ]:

from unsloth import FastLanguageModel
from transformers import TextStreamer
import torch

# Load from Hub (or use local: f"{BASE}/outputs/merged_16bit")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = HUB_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)

TEST = "Given N, count prime numbers ≤ N. Constraints: 1 ≤ N ≤ 10^7, 1s limit."

messages = [
    {"role": "system", "content": "You are an expert competitive programmer. Solve with C++20."},
    {"role": "user",   "content": TEST},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

print("─" * 60)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
_ = model.generate(
    input_ids=inputs, streamer=streamer,
    max_new_tokens=512, temperature=0.2, top_p=0.9,
    repetition_penalty=1.1, do_sample=True,
)

In [ ]:
!python /content/4_app.py --model_path {HUB_ID}